# Coleta `falando_nela` no Google Colab

Template para rodar os coletores em modo producao com Google Drive montado. O codigo fica no Git; os dados completos ficam no Drive.

Se o repositorio estiver privado, crie um Secret no Colab chamado `GITHUB_TOKEN` antes de executar a celula de clone. O token precisa ter permissao de leitura do repositorio.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

os.environ['FALANDO_NELA_DATA_ROOT'] = '/content/drive/MyDrive/falando_nela/data'
Path(os.environ['FALANDO_NELA_DATA_ROOT']).mkdir(parents=True, exist_ok=True)
print(os.environ['FALANDO_NELA_DATA_ROOT'])

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_OWNER = "pedblan"
REPO_NAME = "falando_nela"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.


def get_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def git_env():
    env = os.environ.copy()
    token = get_colab_secret("GITHUB_TOKEN")
    if token:
        env["GITHUB_TOKEN"] = token
        askpass = Path("/tmp/falando_nela_git_askpass.sh")
        askpass.write_text(
            "#!/bin/sh\n"
            "case \"$1\" in\n"
            "  *Username*) echo x-access-token ;;\n"
            "  *Password*) echo \"$GITHUB_TOKEN\" ;;\n"
            "  *) echo \"$GITHUB_TOKEN\" ;;\n"
            "esac\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"
    return env


def run_git(args, cwd=None):
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        env=git_env(),
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(
            "Git falhou. Se o repositorio for privado, crie no Colab um Secret "
            "chamado GITHUB_TOKEN com um token do GitHub com permissao de leitura do repo."
        )
    return result


if not REPO_DIR.exists():
    run_git(["clone", REPO_URL, str(REPO_DIR)])
else:
    run_git(["fetch", "--all", "--tags", "--prune"], cwd=REPO_DIR)
    if not REPO_REF:
        run_git(["pull", "--ff-only"], cwd=REPO_DIR)

if REPO_REF:
    run_git(["checkout", REPO_REF], cwd=REPO_DIR)

run_git(["remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR)
os.chdir(REPO_DIR)
print("Repo em:", Path.cwd())
run_git(["status", "--short"], cwd=REPO_DIR)


## Parametros

Use uma janela curta primeiro para validar escrita no Drive. Depois rode o baseline completo.

In [ ]:
DATA_INICIO = '2011-05-18'
DATA_FIM = '2026-05-18'

# Para validacao inicial no Drive, descomente uma janela curta:
# DATA_INICIO = '2026-02-01'
# DATA_FIM = '2026-02-28'

COLETORES = [
    'coleta.senado.plenario_discursos.collect',
    'coleta.senado.congresso_discursos.collect',
    'coleta.senado.ccj_notas.collect',
    'coleta.camara.plenario_discursos.collect',
    'coleta.camara.ccjc_eventos.collect',
]

In [ ]:
import subprocess
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
manifest_paths = []

for module in COLETORES:
    dataset = module.replace('coleta.', '').replace('.collect', '').replace('.', '-')
    run_id = f'prod-{dataset}-{run_stamp}'
    cmd = [
        'python', '-m', module,
        '--mode', 'prod',
        '--resume',
        '--run-id', run_id,
        '--data-inicio', DATA_INICIO,
        '--data-fim', DATA_FIM,
    ]
    print('Rodando:', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, text=True, capture_output=True)
    print(result.stdout)
    manifest_paths.append(result.stdout.strip().splitlines()[-1])

manifest_paths

In [ ]:
import json
from pathlib import Path

for manifest_path in manifest_paths:
    path = Path(manifest_path)
    manifest = json.loads(path.read_text(encoding='utf-8'))
    print('\nMANIFEST:', path)
    print(json.dumps({
        'source': manifest.get('source'),
        'dataset': manifest.get('dataset'),
        'mode': manifest.get('mode'),
        'sample': manifest.get('sample'),
        'record_counts': manifest.get('record_counts'),
        'partition_counts': manifest.get('partition_counts'),
    }, ensure_ascii=False, indent=2))
    log_path = Path(manifest['log_path'])
    if log_path.exists():
        print('Ultimas linhas do log:')
        print('\n'.join(log_path.read_text(encoding='utf-8').splitlines()[-5:]))